In [7]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, Rescaling, Conv2D, MaxPooling2D

I0000 00:00:1780930129.322720     443 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780930129.324045     443 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780930129.393371     443 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780930132.076853     443 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [8]:
def getDataStreams(data_dir, image_size=(300,300), batch_size=32):

    train_ds = tf.keras.utils.image_dataset_from_directory(
        f"{data_dir}",
        validation_split=0.2,
        batch_size=batch_size,
        subset='training',
        image_size=image_size,
        seed=42  
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        f"{data_dir}",
        validation_split=0.2,
        batch_size=batch_size,
        image_size=image_size,
        subset='validation',
        seed=42
        )

    train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

    return train_ds, val_ds


In [16]:
def buildDetectionModel(input_shape=(300,300,3)):

    input_layer = Input(shape=input_shape)
    x = Rescaling(1./255)(input_layer)
    x = Conv2D(32, (3,3), activation='relu')(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.3)(x)
    x = Conv2D(64, (3,3),activation='relu')(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.3)(x)
    x = Conv2D(128, (3,3), activation='relu')(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.3)(x)
    x = tf.keras.layers.Flatten()(x)
    x = Dense(64, activation='relu')(x)
    output = Dense(1, activation='sigmoid')(x)
    model = tf.keras.models.Model(inputs=input_layer, outputs=output)
    return model

In [17]:
train_ds, val_ds = getDataStreams(data_dir="Untitled Folder/casting_512x512")


Found 1300 files belonging to 2 classes.
Using 1040 files for training.
Found 1301 files belonging to 2 classes.
Using 260 files for validation.


In [18]:
model = buildDetectionModel()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 298, 298, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 149, 149, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 149, 149, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 147, 147, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 73, 73, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 73, 73, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 71, 71, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 35, 35, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 35, 35, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 156800)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │    10,035,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,128,577 (38.64 MB)

 Trainable params: 10,128,577 (38.64 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=10, batch_size=32)

Epoch 1/10
20/33 ━━━━━━━━━━━━━━━━━━━━ 1:04 5s/step - accuracy: 0.4491 - loss: 0.7112

In [15]:
loss, accuracy = model.evaluate(train_ds)
print(loss)
print(accuracy*100)

6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.7674 - loss: 0.4422
0.44223111867904663
76.74418687820435
